# TTF Market Data Validation

This notebook validates the TTF price data used in the Norway TTF event study.

The checks cover the source structure, dates, missing values, numeric fields, OHLC consistency, trading-date gaps and return calculations. The raw CSV is kept unchanged. Cleaning and type conversion are carried out in Python, then the validated dataset is saved for use in the event-alignment notebook.

The aim is simple: establish exactly what data is being used before linking market prices to Gassco announcements.

## Source

**Provider:** Investing.com  
**Market:** ICE Dutch TTF Natural Gas Futures  
**Series:** `TFMBMc1` 
**Source URL:** https://uk.investing.com/commodities/ice-dutch-ttf-gas-c1-futures-historical-data  
**Raw file:** `ice_dutch_ttf_futures_2024-07-01_to_2026-09-17.csv`  
**Observed period:** 1 July 2024 to 17 September 2026  
**Raw rows:** 591

The downloaded CSV is retained as the raw source and is not manually edited.

The file contains daily observations for price, open, high, low, reported volume and percentage change. The exact interpretation of the continuous futures series is discussed separately below because the underlying contract changes through time.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# Resolve the project directory whether Jupyter is opened
# from the project root or the notebooks folder.
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


RAW_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "ttf"
    / "ice_dutch_ttf_futures_2024-07-01_to_2026-09-17.csv"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_FILE = PROCESSED_DIR / "ttf_daily_validated.csv"


if not RAW_FILE.exists():
    raise FileNotFoundError(f"TTF source file not found: {RAW_FILE}")


df_raw = pd.read_csv(RAW_FILE)

print(f"Rows: {len(df_raw):,}")
print(f"Columns: {len(df_raw.columns)}")
print(f"Source: {RAW_FILE}")

display(df_raw.head())

Rows: 591
Columns: 7
Source: data/raw/ttf/ice_dutch_ttf_futures_2024-07-01_to_2026-09-17.csv


,Date,Price,Open,High,Low,Vol.,Change %
0,17/09/2026,78.585,77.75,78.865,75.620,123.11K,1.11%
1,16/09/2026,77.720,77.75,83.280,76.500,0.06K,-2.92%
2,15/09/2026,80.060,82.30,83.415,78.725,147.73K,-3.04%
3,14/09/2026,82.566,82.00,84.500,80.635,136.81K,-0.34%
4,13/09/2026,82.845,82.00,82.845,81.410,1.00K,4.18%


## Schema check

The analysis expects the same seven fields contained in the original download. The schema is checked explicitly so that a change in the source format cannot pass unnoticed into later notebooks.

In [2]:
EXPECTED_COLUMNS = [
    "Date",
    "Price",
    "Open",
    "High",
    "Low",
    "Vol.",
    "Change %",
]

actual_columns = df_raw.columns.tolist()

if actual_columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Unexpected TTF schema.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Found:    {actual_columns}"
    )

print("Schema check: PASS")
print(actual_columns)

Schema check: PASS
['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']


## Date and grain checks

The file should contain one observation for each reported trading date. Dates are parsed using the day/month/year format in the source file, then checked for duplicates.

A duplicated date would make the daily market grain ambiguous and is treated as a validation failure.

In [3]:
trading_dates = pd.to_datetime(
    df_raw["Date"],
    format="%d/%m/%Y",
    errors="raise",
)

duplicate_dates = trading_dates.duplicated(keep=False)

print(f"Parsed dates: {len(trading_dates):,}")
print(f"Unique dates: {trading_dates.nunique():,}")
print(f"Duplicate dates: {duplicate_dates.sum()}")
print(
    f"Observed period: "
    f"{trading_dates.min().date()} to {trading_dates.max().date()}"
)

if duplicate_dates.any():
    display(df_raw.loc[duplicate_dates])
    raise ValueError("Duplicate trading dates found.")

print("\nDate and grain checks: PASS")

Parsed dates: 591
Unique dates: 591
Duplicate dates: 0
Observed period: 2024-07-01 to 2026-09-17

Date and grain checks: PASS


## Missing values

Missing values are checked before any rows are filtered or transformed. Rows are not dropped automatically because doing so would change the market series without first identifying the problem.

In [4]:
missing_values = df_raw.isna().sum()

print(missing_values)

if missing_values.sum() > 0:
    missing_rows = df_raw.loc[df_raw.isna().any(axis=1)]
    display(missing_rows)

    raise ValueError(
        f"Found {missing_values.sum()} missing values in the raw TTF data."
    )

print("\nMissing-value check: PASS")

Date        0
Price       0
Open        0
High        0
Low         0
Vol.        0
Change %    0
dtype: int64

Missing-value check: PASS


## Numeric price fields

`Price`, `Open`, `High` and `Low` are required for the market analysis and should all be numeric.

Each field is parsed explicitly. Any value that cannot be converted is treated as a validation failure rather than being replaced or removed.

In [5]:
PRICE_COLUMNS = ["Price", "Open", "High", "Low"]

numeric_prices = df_raw[PRICE_COLUMNS].apply(
    pd.to_numeric,
    errors="coerce",
)

invalid_numeric = numeric_prices.isna().sum()

print("Invalid numeric values:")
print(invalid_numeric)

if invalid_numeric.sum() > 0:
    invalid_rows = df_raw.loc[
        numeric_prices.isna().any(axis=1),
        ["Date", *PRICE_COLUMNS],
    ]

    display(invalid_rows)

    raise ValueError(
        f"Found {invalid_numeric.sum()} invalid values "
        "across the OHLC price fields."
    )

print("\nNumeric price check: PASS")

Invalid numeric values:
Price    0
Open     0
High     0
Low      0
dtype: int64

Numeric price check: PASS


## OHLC consistency

The reported daily price fields should be internally consistent.

For each row, `High` must be at least as large as `Open`, `Low` and `Price`. `Low` must be no greater than those values. A violation would indicate an internally inconsistent market observation.

In [6]:
invalid_ohlc = (
    (numeric_prices["High"] < numeric_prices["Low"])
    | (numeric_prices["Open"] > numeric_prices["High"])
    | (numeric_prices["Open"] < numeric_prices["Low"])
    | (numeric_prices["Price"] > numeric_prices["High"])
    | (numeric_prices["Price"] < numeric_prices["Low"])
)

invalid_ohlc_count = int(invalid_ohlc.sum())

print(f"OHLC inconsistencies: {invalid_ohlc_count}")

if invalid_ohlc_count > 0:
    display(
        df_raw.loc[
            invalid_ohlc,
            ["Date", "Price", "Open", "High", "Low"],
        ]
    )

    raise ValueError(
        f"Found {invalid_ohlc_count} internally inconsistent OHLC rows."
    )

print("OHLC consistency check: PASS")

OHLC inconsistencies: 0
OHLC consistency check: PASS


## Reported volume and percentage change

The source stores `Vol.` and `Change %` as text rather than numeric fields.

Their formats are checked across the full dataset before conversion. The volume field uses a `K` suffix throughout this file, while percentage changes use `%`.

The volume field is converted to its numerical equivalent as reported by the vendor. The percentage-change field is converted to decimal form and retained only as a check against returns calculated independently from `Price`.

In [7]:
volume_text = df_raw["Vol."].astype(str).str.strip()
change_text = df_raw["Change %"].astype(str).str.strip()

valid_volume_format = volume_text.str.fullmatch(
    r"\d+(\.\d+)?K"
)

valid_change_format = change_text.str.fullmatch(
    r"-?\d+(\.\d+)?%"
)

print(
    "Invalid volume formats:",
    int((~valid_volume_format).sum()),
)

print(
    "Invalid percentage-change formats:",
    int((~valid_change_format).sum()),
)

if not valid_volume_format.all():
    display(
        df_raw.loc[
            ~valid_volume_format,
            ["Date", "Vol."],
        ]
    )
    raise ValueError("Unexpected format found in Vol.")

if not valid_change_format.all():
    display(
        df_raw.loc[
            ~valid_change_format,
            ["Date", "Change %"],
        ]
    )
    raise ValueError("Unexpected format found in Change %.")

print("\nAuxiliary field format checks: PASS")

Invalid volume formats: 0
Invalid percentage-change formats: 0

Auxiliary field format checks: PASS


In [8]:
reported_volume = (
    volume_text
    .str.removesuffix("K")
    .astype(float)
    .mul(1_000)
)

reported_change = (
    change_text
    .str.removesuffix("%")
    .astype(float)
    .div(100)
)

if (reported_volume < 0).any():
    raise ValueError("Negative reported volume found.")

if not np.isfinite(reported_change).all():
    raise ValueError("Non-finite percentage change found.")

print("Auxiliary field conversion: PASS")

display(
    pd.DataFrame(
        {
            "raw_volume": df_raw["Vol."].head(),
            "reported_volume": reported_volume.head(),
            "raw_change": df_raw["Change %"].head(),
            "reported_change": reported_change.head(),
        }
    )
)

Auxiliary field conversion: PASS


,raw_volume,reported_volume,raw_change,reported_change
0,123.11K,123110.0,1.11%,0.0111
1,0.06K,60.0,-2.92%,-0.0292
2,147.73K,147730.0,-3.04%,-0.0304
3,136.81K,136810.0,-0.34%,-0.0034
4,1.00K,1000.0,4.18%,0.0418


## Trading date gaps

The source does not contain an observation for every calendar day. The spacing between consecutive reported TTF dates is therefore checked before event alignment.

A gap is not treated automatically as missing data. Exchange closures, weekends and the vendor's session convention can all produce gaps between observations. No dates are inserted or forward-filled.

Longer gaps are listed for review because they may affect how a Gassco announcement is mapped to the next available market observation.

In [9]:
sorted_dates = (
    pd.Series(trading_dates)
    .sort_values()
    .reset_index(drop=True)
)

calendar_gap_days = sorted_dates.diff().dt.days

if (calendar_gap_days.dropna() <= 0).any():
    raise ValueError(
        "Non-positive date spacing found after sorting."
    )

print("Calendar-day gap frequencies:")
print(
    calendar_gap_days
    .value_counts()
    .sort_index()
)

print(
    "\nLargest observed gap:",
    int(calendar_gap_days.max()),
    "calendar days",
)

gap_table = pd.DataFrame(
    {
        "trading_date": sorted_dates,
        "previous_trading_date": sorted_dates.shift(1),
        "calendar_gap_days": calendar_gap_days,
    }
)

long_gaps = gap_table.loc[
    gap_table["calendar_gap_days"] > 3
].copy()

print(
    f"\nGaps greater than 3 calendar days: "
    f"{len(long_gaps)}"
)

display(long_gaps)

Calendar-day gap frequencies:
Date
1.0    472
2.0     24
3.0     91
5.0      3
Name: count, dtype: int64

Largest observed gap: 5 calendar days

Gaps greater than 3 calendar days: 3


,trading_date,previous_trading_date,calendar_gap_days
206,2025-04-22,2025-04-17,5.0
383,2025-12-29,2025-12-24,5.0
451,2026-04-07,2026-04-02,5.0


## Validated TTF table

The validated fields are now assembled into the market dataset used by the rest of the project.

The table is sorted by trading date and contains one row for each observed market date. The raw source remains unchanged.

In [10]:
df_clean = pd.DataFrame(
    {
        "trading_date": trading_dates,
        "price": numeric_prices["Price"],
        "open": numeric_prices["Open"],
        "high": numeric_prices["High"],
        "low": numeric_prices["Low"],
        "reported_volume": reported_volume,
        "reported_change": reported_change,
    }
)

df_clean = (
    df_clean
    .sort_values("trading_date")
    .reset_index(drop=True)
)

print(f"Rows: {len(df_clean):,}")
print(f"Unique trading dates: {df_clean['trading_date'].nunique():,}")
print(
    "Observed period:",
    df_clean["trading_date"].min().date(),
    "to",
    df_clean["trading_date"].max().date(),
)

display(df_clean.head())

Rows: 591
Unique trading dates: 591
Observed period: 2024-07-01 to 2026-09-17


,trading_date,price,open,high,low,reported_volume,reported_change
0,2024-07-01,33.485,34.455,35.110,33.400,68960.0,-0.0289
1,2024-07-02,33.685,33.435,33.900,33.115,69290.0,0.0060
2,2024-07-03,32.691,33.500,33.745,32.620,96040.0,-0.0295
3,2024-07-04,33.246,32.845,33.800,32.285,69880.0,0.0170
4,2024-07-05,33.067,33.555,33.975,32.875,55060.0,-0.0054


## Market instrument

The market data are taken from the Investing.com series `TFMBMc1`, which is labelled as ICE Dutch TTF Natural Gas Futures and reported in euros.

The series does not represent one fixed delivery-month contract across the full sample. The displayed contract changes through time as the prompt contract rolls forward. This matters because a price change around a contract transition can reflect both market movement and a change in the underlying delivery month.

The exact historical roll rule used by the data provider has not been established from the available documentation. The series is therefore treated as a vendor-provided prompt TTF futures series rather than as a fixed contract or an official ICE settlement series.

The source field `Price` is retained as `price`. It is not relabelled as an ICE settlement price because the source does not establish that equivalence.

## Price-return reconciliation

Simple returns are calculated between consecutive observed market dates and compared with the percentage changes reported by the source.

This is an internal consistency check. It tests whether the ordering and numerical interpretation of the price series agree with the vendor-reported daily change. It does not validate the underlying contract roll method or establish that `price` is an official ICE settlement.

In [11]:
df_clean["simple_return"] = (
    df_clean["price"]
    .pct_change(fill_method=None)
)

df_clean["return_difference"] = (
    df_clean["simple_return"]
    - df_clean["reported_change"]
)

comparison = df_clean.loc[
    df_clean["simple_return"].notna(),
    [
        "trading_date",
        "price",
        "simple_return",
        "reported_change",
        "return_difference",
    ],
].copy()

mean_abs_difference = (
    comparison["return_difference"]
    .abs()
    .mean()
)

max_abs_difference = (
    comparison["return_difference"]
    .abs()
    .max()
)

print(f"Comparable observations: {len(comparison):,}")
print(f"Mean absolute difference: {mean_abs_difference:.8f}")
print(f"Maximum absolute difference: {max_abs_difference:.8f}")

ROUNDING_TOLERANCE = 0.000051

if max_abs_difference > ROUNDING_TOLERANCE:
    raise ValueError(
        "Price-derived returns do not reconcile with "
        "the vendor-reported percentage change."
    )

print("\nReturn reconciliation: PASS")

Comparable observations: 590
Mean absolute difference: 0.00002422
Maximum absolute difference: 0.00004967

Return reconciliation: PASS


### Reconciliation result

The calculated returns closely match the vendor-reported percentage changes across the 590 comparable observations.

The remaining differences are consistent with rounding in the reported percentage-change field. Returns used in the event study are therefore calculated directly from `price`, while `reported_change` is kept only as a validation field.

## Log returns

Log returns are used because they are additive across consecutive observed price intervals. The first observation has no previous price and therefore has no return.

For consecutive observations:

$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$

where \(P_t\) is the price on the current observed market date and \(P_{t-1}\) is the price on the previous observed market date.


In [12]:
if (df_clean["price"] <= 0).any():
    raise ValueError(
        "Log returns cannot be calculated from non-positive prices."
    )

df_clean["log_return"] = (
    np.log(df_clean["price"])
    .diff()
)

expected_missing_returns = 1
actual_missing_returns = int(
    df_clean["log_return"].isna().sum()
)

if actual_missing_returns != expected_missing_returns:
    raise ValueError(
        "Unexpected number of missing log returns. "
        f"Expected 1, found {actual_missing_returns}."
    )

print("Log-return calculation: PASS")

display(
    df_clean[
        [
            "trading_date",
            "price",
            "simple_return",
            "log_return",
        ]
    ].head()
)

Log-return calculation: PASS


,trading_date,price,simple_return,log_return
0,2024-07-01,33.485,NaN,NaN
1,2024-07-02,33.685,0.005973,0.005955
2,2024-07-03,32.691,-0.029509,-0.029953
3,2024-07-04,33.246,0.016977,0.016835
4,2024-07-05,33.067,-0.005384,-0.005399


## Save validated dataset

The validated market table is saved as the single TTF input for downstream notebooks.

This prevents the cleaning and return calculations from being repeated independently in the event-alignment analysis.

In [13]:
OUTPUT_COLUMNS = [
    "trading_date",
    "price",
    "open",
    "high",
    "low",
    "reported_volume",
    "reported_change",
    "simple_return",
    "log_return",
]

ttf_validated = df_clean[OUTPUT_COLUMNS].copy()

if len(ttf_validated) != len(df_raw):
    raise ValueError("Row count changed during TTF preparation.")

if not ttf_validated["trading_date"].is_unique:
    raise ValueError("Validated TTF dataset contains duplicate trading dates.")

if ttf_validated["trading_date"].isna().any():
    raise ValueError("Validated TTF dataset contains missing trading dates.")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ttf_validated.to_csv(
    OUTPUT_FILE,
    index=False,
)

print(f"Saved: {OUTPUT_FILE}")
print(f"Rows: {len(ttf_validated):,}")
print(f"Columns: {len(ttf_validated.columns)}")

Saved: data/processed/ttf_daily_validated.csv
Rows: 591
Columns: 9


## Validation result

The source contains 591 unique observations from 1 July 2024 to 17 September 2026. No missing values, duplicate dates, invalid OHLC values or malformed numeric fields were identified.

Returns calculated from `price` reconcile with the source-reported percentage changes within the expected rounding tolerance.

The validated dataset is saved for downstream event alignment.

The main remaining market-data limitation is the use of a vendor-provided prompt futures series whose historical contract-roll methodology has not been independently established.